# Mistral-7B-Instruct-v0.2 topic labelling (K=65)

Kaggle kernel used for the paper's topic labels (`mistral_labels_v3.json`). Input: `topic_words_v3.json`
(10 c-TF-IDF words per topic). Prompt, 4-bit NF4 loading and greedy generation settings are in the cells below
and in `MISTRAL_PROMPT_AND_SETTINGS.md`. This notebook is a reconstruction of the kernel's version 2;
the run produced 65 labels, 0 flagged (raw == final).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes', 'accelerate'], check=True)
print('deps ok')

In [ ]:
import os, json, re, time, glob

print('ls /kaggle/input:', os.listdir('/kaggle/input'))
matches = glob.glob('/kaggle/input/**/topic_words_v3.json', recursive=True)
print('found topic_words_v3.json at:', matches)
assert matches, 'topic_words_v3.json not found under /kaggle/input -- check dataset attachment'
with open(matches[0]) as f:
    topic_words = json.load(f)

topic_ids = sorted(topic_words.keys(), key=lambda x: int(x))
print(f'{len(topic_ids)} topics loaded')

MISTRAL_MODEL_ID = 'unsloth/mistral-7b-instruct-v0.2'
MISTRAL_PROMPT = (
    "[INST] Topic keywords: [KEYWORDS]\n"
    "Give a short label (max 5 words) for this topic. Return only the label. [/INST]"
)

# Refusal-detection patterns (phrases seen in refusals of earlier Mistral labelling runs).
REFUSAL_PATTERNS = [
    r"do(?:es)?\s+not\s+match",
    r"does not contain any keyword",
    r"should be related to",
    r"please check the keyword",
    r"i (?:cannot|can't|do not have access|don't have access)",
    r"i'm sorry",
    r"as an ai",
    r"not related to (?:the|any)",
]
REFUSAL_RE = re.compile('|'.join(REFUSAL_PATTERNS), re.IGNORECASE)

def looks_like_refusal(label):
    return bool(REFUSAL_RE.search(label))

print('setup ok')

In [ ]:
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print('Loading', MISTRAL_MODEL_ID, 'in 4-bit...')
t0 = time.time()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
generator = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    task='text-generation',
    temperature=0.1,
    max_new_tokens=25,
    do_sample=False,
    repetition_penalty=1.1,
    truncation=True,
    return_full_text=False,
)
print(f'model loaded in {time.time()-t0:.1f}s')

In [ ]:
raw_labels = {}
final_labels = {}
flagged = {}

t0 = time.time()
for tid in topic_ids:
    words = [w for w, _ in topic_words[tid]]
    keywords = ', '.join(words)
    prompt = MISTRAL_PROMPT.replace('[KEYWORDS]', keywords)
    out = generator(prompt)
    raw = out[0]['generated_text'].strip().split('\n')[0].strip()
    raw_labels[tid] = raw

    if looks_like_refusal(raw) or len(raw) == 0:
        fallback = ', '.join(words[:5])
        final_labels[tid] = fallback
        flagged[tid] = {'raw_mistral_output': raw, 'fallback_used': fallback, 'reason': 'matched refusal pattern' if raw else 'empty output'}
        print(f'[T{tid}] FLAGGED -- raw: {raw!r} -> fallback: {fallback!r}')
    else:
        final_labels[tid] = raw
        print(f'[T{tid}] ok: {raw!r}')

print(f"\nDone in {time.time()-t0:.1f}s. Flagged: {len(flagged)}/{len(topic_ids)}")

In [ ]:
os.makedirs('/kaggle/working/out', exist_ok=True)
with open('/kaggle/working/out/mistral_labels_v3.json', 'w') as f:
    json.dump(final_labels, f, indent=2, ensure_ascii=False)
with open('/kaggle/working/out/mistral_labels_v3_raw.json', 'w') as f:
    json.dump(raw_labels, f, indent=2, ensure_ascii=False)
with open('/kaggle/working/out/mistral_labels_v3_flagged.json', 'w') as f:
    json.dump(flagged, f, indent=2, ensure_ascii=False)
print('saved. flagged topics:', list(flagged.keys()))
print('\nFinal labels:')
for tid in topic_ids:
    print(f'  T{tid}: {final_labels[tid]}')